### Notebook 2 - Data Engineering
- transform the raw Brisbane bikeway counts dataset into a clean, tidy dataset suitable for EDA and ML.

1. Load the raw datasets.
2. Reshape the bikeway counts from wide format to long format.
3. Extract the Site ID and Transport Mode from the column names.
4. Handle missing values.
5. Merge with the counter location metadata.
6. Save the cleaned dataset for future analysis.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [2]:
counts = pd.read_csv("../data/raw/bikeway-counts.csv")
locations = pd.read_csv("../data/raw/bikeway-counts-counter-locations.csv")

In [3]:
counts.head()

,Date,A001 Cyclist,A001 Pedestrian,A002 Cyclist,A002 Pedestrian,A003 Cyclist,A003 Pedestrian,A004E Cyclist,A004N Pedestrian,A005E Pedestrian,A005W Cyclist,A006 Cyclist,A006 Pedestrian,A007 Cyclist,A007 Pedestrian,A008 Cyclist,A008 Pedestrian,A009 Cyclist,A009 Pedestrian,A010 Cyclist,A010 Pedestrian,A011 Cyclist,A011 Pedestrian,A012 Cyclist,A012 Pedestrian,A013 Cyclist,A013 Pedestrian,A014 Cyclist,A014 Pedestrian,A016E Cyclist,A016E Pedestrian,A016W Cyclist,A016W Pedestrian,A017 Cyclist,A017 Pedestrian,A018 Cyclist,A018 Pedestrian,A019 Cyclist,A019 Pedestrian,A020 Cyclist,A020 Pedestrian,A021 Cyclist,A021 Pedestrian,A022 Cyclist,A023 Cyclist,A023 Pedestrian,A024 Cyclist,A024 Pedestrian,A025 Cyclist,A025 Pedestrian,A026 Cyclist,A027 Cyclist,A028 Cyclist,A028 Pedestrian,A029 Cyclist,A030 Cyclist,A030 Pedestrian,A032 Cyclist,A032 Pedestrian,A032 Scooter,A103 Cyclist,A103 Pedestrian,A200 Cyclist,A200 Scooter,A201 Cyclist,A202 Cyclist,A202 Scooter,A203 Cyclist,A203 Scooter,A204 Cyclist,A204 Scooter
0,2022-01-01,NaN,NaN,38.0,698.0,120.0,521.0,270.0,806.0,454.0,NaN,NaN,NaN,142.0,310,NaN,NaN,243.0,463.0,85.0,216.0,129.0,NaN,91.0,152.0,206.0,1277.0,NaN,NaN,80.0,531.0,163.0,1257.0,3.0,268.0,174.0,673.0,NaN,NaN,2.0,84.0,209.0,1435.0,220.0,596.0,2581.0,85.0,304.0,761.0,4023.0,166,146.0,261.0,687.0,65.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,134.0,158.0,256.0,51.0,282.0,NaN,NaN
1,2022-01-02,NaN,NaN,156.0,591.0,402.0,484.0,880.0,782.0,616.0,NaN,NaN,NaN,829.0,724,NaN,NaN,1250.0,1106.0,240.0,380.0,607.0,NaN,473.0,335.0,1051.0,2088.0,NaN,NaN,622.0,636.0,363.0,889.0,17.0,814.0,883.0,929.0,NaN,NaN,50.0,176.0,1301.0,2514.0,616.0,2382.0,4426.0,589.0,552.0,2793.0,6457.0,436,525.0,1106.0,1019.0,349.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,236.0,304.0,343.0,63.0,485.0,NaN,NaN
2,2022-01-03,NaN,NaN,152.0,614.0,402.0,349.0,888.0,692.0,582.0,NaN,NaN,NaN,801.0,600,NaN,NaN,1160.0,1006.0,310.0,340.0,638.0,NaN,403.0,374.0,993.0,1971.0,NaN,NaN,582.0,527.0,328.0,591.0,19.0,NaN,695.0,910.0,NaN,NaN,33.0,149.0,1000.0,2267.0,592.0,2149.0,3641.0,465.0,535.0,2599.0,6142.0,441,542.0,1025.0,1016.0,334.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,183.0,277.0,332.0,114.0,402.0,NaN,NaN
3,2022-01-04,NaN,NaN,158.0,514.0,281.0,215.0,1041.0,681.0,446.0,NaN,NaN,NaN,578.0,399,NaN,NaN,790.0,894.0,241.0,255.0,651.0,NaN,401.0,356.0,644.0,1476.0,NaN,NaN,463.0,445.0,321.0,367.0,26.0,NaN,673.0,812.0,NaN,NaN,21.0,134.0,712.0,1860.0,646.0,1538.0,2664.0,498.0,549.0,2423.0,4298.0,434,491.0,688.0,784.0,294.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,286.0,373.0,399.0,157.0,405.0,NaN,NaN
4,2022-01-05,NaN,NaN,111.0,454.0,254.0,185.0,1011.0,766.0,475.0,NaN,NaN,NaN,588.0,462,NaN,NaN,823.0,913.0,183.0,199.0,572.0,NaN,362.0,272.0,481.0,1370.0,NaN,NaN,464.0,263.0,303.0,487.0,24.0,NaN,645.0,743.0,NaN,NaN,25.0,125.0,608.0,1886.0,618.0,1416.0,2604.0,441.0,454.0,2341.0,4089.0,448,477.0,727.0,766.0,292.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,237.0,361.0,401.0,157.0,396.0,NaN,NaN


In [15]:
#convert it to long type - using melt
long_df = counts.melt(
    id_vars="Date",
    var_name="Counter",
    value_name="Count"
)

In [16]:
long_df.head(20)

,Date,Counter,Count
0,2022-01-01,A001 Cyclist,NaN
1,2022-01-02,A001 Cyclist,NaN
2,2022-01-03,A001 Cyclist,NaN
3,2022-01-04,A001 Cyclist,NaN
4,2022-01-05,A001 Cyclist,NaN
5,2022-01-06,A001 Cyclist,NaN
6,2022-01-07,A001 Cyclist,NaN
7,2022-01-08,A001 Cyclist,NaN
8,2022-01-09,A001 Cyclist,NaN
9,2022-01-10,A001 Cyclist,NaN


In [17]:
print("Original shape:", counts.shape)
print("New shape:", long_df.shape)

Original shape: (1461, 71)
New shape: (102270, 3)


In [18]:
#split site id and transport from counter
long_df[["Site_ID", "Mode"]] = long_df["Counter"].str.split(
    " ",
    n=1,
    expand=True
)

In [19]:
long_df.head(20)

,Date,Counter,Count,Site_ID,Mode
0,2022-01-01,A001 Cyclist,NaN,A001,Cyclist
1,2022-01-02,A001 Cyclist,NaN,A001,Cyclist
2,2022-01-03,A001 Cyclist,NaN,A001,Cyclist
3,2022-01-04,A001 Cyclist,NaN,A001,Cyclist
4,2022-01-05,A001 Cyclist,NaN,A001,Cyclist
5,2022-01-06,A001 Cyclist,NaN,A001,Cyclist
6,2022-01-07,A001 Cyclist,NaN,A001,Cyclist
7,2022-01-08,A001 Cyclist,NaN,A001,Cyclist
8,2022-01-09,A001 Cyclist,NaN,A001,Cyclist
9,2022-01-10,A001 Cyclist,NaN,A001,Cyclist


In [20]:
long_df = long_df.drop(columns=["Counter"])

In [21]:
print(long_df.shape)

(102270, 4)


In [22]:
long_df["Count"].isna().sum()

np.int64(44518)

In [24]:
round(long_df["Count"].isna().mean() * 100,2)

np.float64(43.53)

In [25]:
#what percentage of each site's data is missing?
long_df.groupby("Site_ID")["Count"].apply(lambda x: x.isna().mean()*100).sort_values(ascending=False)

Site_ID
A201     97.467488
A003     90.417522
A011     86.550308
A103     83.025325
A002     76.009582
A032     73.853525
A009     70.396988
A019     68.856947
A204     67.145791
A203     65.297741
A005W    58.521561
A006     55.612594
A030     54.140999
A202     51.882272
A200     49.452430
A004E    45.585216
A024     43.668720
A016E    43.326489
A018     42.402464
A017     40.041068
A025     38.843258
A010     38.843258
A001     38.603696
A028     37.337440
A008     36.926762
A005E    36.481862
A013     25.804244
A023     22.895277
A016W    17.522245
A012     16.153320
A004N    15.742642
A014      9.582478
A022      7.939767
A021      6.502396
A020      2.840520
A007      1.471595
A029      0.547570
A027      0.136893
A026      0.000000
Name: Count, dtype: float64

In [26]:
#missing values by transport
long_df.groupby("Mode")["Count"].apply(lambda x: x.isna().mean()*100)

Mode
Cyclist       39.737684
Pedestrian    45.328542
Scooter       61.519507
Name: Count, dtype: float64

In [27]:
long_df[long_df["Count"].isna()].head(20)

,Date,Count,Site_ID,Mode
0,2022-01-01,NaN,A001,Cyclist
1,2022-01-02,NaN,A001,Cyclist
2,2022-01-03,NaN,A001,Cyclist
3,2022-01-04,NaN,A001,Cyclist
4,2022-01-05,NaN,A001,Cyclist
5,2022-01-06,NaN,A001,Cyclist
6,2022-01-07,NaN,A001,Cyclist
7,2022-01-08,NaN,A001,Cyclist
8,2022-01-09,NaN,A001,Cyclist
9,2022-01-10,NaN,A001,Cyclist


#### Handling Missing Values
Approximately 44% missing values - mostly structural

- Certain monitoring sites have consistently missing observations.
- Missingness varies by transport mode.
- Many sites have continuous missing values across the entire observation period.

These missing values most likely indicate that a monitoring site did not record a particular transport mode rather than missing sensor readings.

Since 'Count` is the target variable for analysis and prediction, rows with missing counts were removed from the analytical dataset.

In [30]:
long_df = long_df.dropna(subset=["Count"]).reset_index(drop=True)

In [31]:
print(long_df.shape)

print(long_df["Count"].isna().sum())

(57752, 4)
0


In [32]:
long_df.head(20)

,Date,Count,Site_ID,Mode
0,2022-07-03,1246.0,A001,Cyclist
1,2022-07-04,1573.0,A001,Cyclist
2,2022-07-05,610.0,A001,Cyclist
3,2022-07-06,1350.0,A001,Cyclist
4,2022-07-07,2399.0,A001,Cyclist
5,2022-07-08,2333.0,A001,Cyclist
6,2022-07-09,2051.0,A001,Cyclist
7,2022-07-10,2138.0,A001,Cyclist
8,2022-07-11,2093.0,A001,Cyclist
9,2022-07-12,2086.0,A001,Cyclist


In [33]:
long_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 57752 entries, 0 to 57751
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Date     57752 non-null  str    
 1   Count    57752 non-null  float64
 2   Site_ID  57752 non-null  str    
 3   Mode     57752 non-null  str    
dtypes: float64(1), str(3)
memory usage: 3.0 MB


In [34]:
locations["Type"].value_counts()

Type
Manual       196
Automatic     61
Name: count, dtype: int64

In [35]:
locations[locations["Type"] == "Automatic"].head()

,Site ID,Parent Site ID,Site Name,Short Name,Type,Latitude,Longitude,Start_Date,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,geolocation
67,A001,A001,"Bicentennial Bikeway, Auchenflower","Bicentennial, Auchenflower",Automatic,-27.478042,153.000350,2013.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,"-27.478042, 153.00035"
68,A003,A003,"Ekibin Park, Greenslopes","Ekibin Park, Greenslopes",Automatic,-27.507232,153.041925,2015.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,"-27.50723217, 153.0419253"
69,A004E,A004,"Eleanor Schonell Br Cyclists, St Lucia","Eleanor Schonell Br, St Lucia",Automatic,-27.498161,153.024863,2013.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,"-27.49816128, 153.0248634"
70,A009,A009,"Kedron Brook Bikeway, Lutwyche","Kedron Br B'way, Lutwyche",Automatic,-27.418186,153.032104,2014.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,"-27.418186, 153.032104"
71,A013,A013,"Riverwalk, Newfarm","Riverwalk, New Farm",Automatic,-27.464029,153.038670,2014.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,"-27.464029, 153.03867"


In [36]:
automatic_locations = locations[
    locations["Type"] == "Automatic"
].copy()

In [37]:
location_clean = automatic_locations[
    [
        "Site ID",
        "Site Name",
        "Latitude",
        "Longitude",
        "Type",
        "Start_Date"
    ]
].copy()

In [40]:
location_clean.head()

,Site ID,Site Name,Latitude,Longitude,Type,Start_Date
67,A001,"Bicentennial Bikeway, Auchenflower",-27.478042,153.000350,Automatic,2013.0
68,A003,"Ekibin Park, Greenslopes",-27.507232,153.041925,Automatic,2015.0
69,A004E,"Eleanor Schonell Br Cyclists, St Lucia",-27.498161,153.024863,Automatic,2013.0
70,A009,"Kedron Brook Bikeway, Lutwyche",-27.418186,153.032104,Automatic,2014.0
71,A013,"Riverwalk, Newfarm",-27.464029,153.038670,Automatic,2014.0


In [41]:
print("Unique Site IDs in counts :", long_df["Site_ID"].nunique())
print("Unique Site IDs in locations :", location_clean["Site ID"].nunique())

Unique Site IDs in counts : 39
Unique Site IDs in locations : 61


In [44]:
#check for unmatched SiteIDs - any siteids existing in long_df that are not in location_clean
missing_locations = set(long_df["Site_ID"]) - set(location_clean["Site ID"])
missing_locations

set()

#### Merge the datasets to add location details

In [45]:
master_df = long_df.merge(
    location_clean,
    left_on="Site_ID",
    right_on="Site ID",
    how="left"
)

In [46]:
master_df.head()

,Date,Count,Site_ID,Mode,Site ID,Site Name,Latitude,Longitude,Type,Start_Date
0,2022-07-03,1246.0,A001,Cyclist,A001,"Bicentennial Bikeway, Auchenflower",-27.478042,153.00035,Automatic,2013.0
1,2022-07-04,1573.0,A001,Cyclist,A001,"Bicentennial Bikeway, Auchenflower",-27.478042,153.00035,Automatic,2013.0
2,2022-07-05,610.0,A001,Cyclist,A001,"Bicentennial Bikeway, Auchenflower",-27.478042,153.00035,Automatic,2013.0
3,2022-07-06,1350.0,A001,Cyclist,A001,"Bicentennial Bikeway, Auchenflower",-27.478042,153.00035,Automatic,2013.0
4,2022-07-07,2399.0,A001,Cyclist,A001,"Bicentennial Bikeway, Auchenflower",-27.478042,153.00035,Automatic,2013.0


#### Validating the merge

In [47]:
print("Rows before merge :", len(long_df))
print("Rows after merge  :", len(master_df))

Rows before merge : 57752
Rows after merge  : 57752


In [48]:
master_df["Site Name"].isna().sum()

np.int64(0)

In [49]:
master_df["Latitude"].isna().sum()

np.int64(0)

In [50]:
master_df["Longitude"].isna().sum()

np.int64(0)

In [51]:
#remove the duplicate site id columns
master_df = master_df.drop(columns=["Site ID"])

In [52]:
master_df = master_df[
    [
        "Date",
        "Site_ID",
        "Site Name",
        "Mode",
        "Count",
        "Latitude",
        "Longitude",
        "Type",
        "Start_Date",
    ]
]

In [54]:
#final required dataset for analysis
master_df.head()

,Date,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date
0,2022-07-03,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013.0
1,2022-07-04,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013.0
2,2022-07-05,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013.0
3,2022-07-06,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013.0
4,2022-07-07,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013.0


#### Adding/Modifying relevant features to master_df

In [60]:
master_df["Start_Date"] = master_df["Start_Date"].astype(int)

In [61]:
master_df["Start_Date"].dtype

dtype('int64')

In [62]:
master_df["Date"] = pd.to_datetime(master_df["Date"])

In [63]:
master_df["Year"] = master_df["Date"].dt.year

In [64]:
master_df["Month"] = master_df["Date"].dt.month

In [65]:
master_df["Month_Name"] = master_df["Date"].dt.month_name()

In [66]:
master_df["Day"] = master_df["Date"].dt.day

In [67]:
master_df["Day_of_Week"] = master_df["Date"].dt.day_name()

In [68]:
master_df["Weekday_Weekend"] = np.where(
    master_df["Date"].dt.dayofweek >= 5,
    "Weekend",
    "Weekday"
)

In [69]:
master_df = master_df[
    [
        "Date",
        "Year",
        "Month",
        "Month_Name",
        "Day",
        "Day_of_Week",
        "Weekday_Weekend",
        "Site_ID",
        "Site Name",
        "Mode",
        "Count",
        "Latitude",
        "Longitude",
        "Type",
        "Start_Date",
    ]
]

In [70]:
master_df.head()

,Date,Year,Month,Month_Name,Day,Day_of_Week,Weekday_Weekend,Site_ID,Site Name,Mode,Count,Latitude,Longitude,Type,Start_Date
0,2022-07-03,2022,7,July,3,Sunday,Weekend,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1246.0,-27.478042,153.00035,Automatic,2013
1,2022-07-04,2022,7,July,4,Monday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1573.0,-27.478042,153.00035,Automatic,2013
2,2022-07-05,2022,7,July,5,Tuesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,610.0,-27.478042,153.00035,Automatic,2013
3,2022-07-06,2022,7,July,6,Wednesday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,1350.0,-27.478042,153.00035,Automatic,2013
4,2022-07-07,2022,7,July,7,Thursday,Weekday,A001,"Bicentennial Bikeway, Auchenflower",Cyclist,2399.0,-27.478042,153.00035,Automatic,2013


In [71]:
print("=" * 50)
print("DATA QUALITY REPORT")
print("=" * 50)

print(f"Rows: {len(master_df):,}")
print(f"Columns: {master_df.shape[1]}")
print(f"Unique Sites: {master_df['Site_ID'].nunique()}")
print(f"Date Range: {master_df['Date'].min()} to {master_df['Date'].max()}")

print("\nMissing Values")
print(master_df.isna().sum())

print("\nDuplicate Rows:", master_df.duplicated().sum())

DATA QUALITY REPORT
Rows: 57,752
Columns: 15
Unique Sites: 39
Date Range: 2022-01-01 00:00:00 to 2025-12-31 00:00:00

Missing Values
Date               0
Year               0
Month              0
Month_Name         0
Day                0
Day_of_Week        0
Weekday_Weekend    0
Site_ID            0
Site Name          0
Mode               0
Count              0
Latitude           0
Longitude          0
Type               0
Start_Date         0
dtype: int64

Duplicate Rows: 0


In [72]:
master_df.to_csv(
    "../data/processed/master_dataset.csv",
    index=False
)